In [1]:
# =========================
# LIMPIEZA DE ENTORNO
# =========================

!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu protobuf transformers peft accelerate -q

# =========================
# INSTALACIÓN BASE (COMPATIBLE)
# =========================

!pip install transformers==4.38.2 accelerate==0.27.2 protobuf==3.20.3 -q

# =========================
# MÉTRICAS
# =========================

!pip install evaluate bert-score sacrebleu unbabel-comet bert-score -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 3.20.3 which is incompatible.
google-cloud-datastore 2.25.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataplex 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.3 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-storage 2.39.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataproc 5.28.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires

In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [3]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

# SILABIFICADOR

In [4]:
#Silabificador v2, cambio en la forma de impresión de vacíos

VOWELS = {'a', 'e', 'i', 'o'}

DIGRAPHS = {'ch', 'sh', 'ts'}

CODAS = {'n', 's', 'sh', 'x'}

def tokenize(word):
    tokens = []
    i = 0

    while i < len(word):

        if i + 1 < len(word) and word[i:i+2] in DIGRAPHS:
            tokens.append(word[i:i+2])
            i += 2
        else:
            tokens.append(word[i])
            i += 1

    return tokens

def silabificador_shipibo(word):

    tokens = tokenize(word)

    syllables = []
    current = []

    i = 0

    while i < len(tokens):

        current.append(tokens[i])

        # si es vocal
        if tokens[i] in VOWELS:

            # mirar siguiente
            if i + 1 < len(tokens):

                nxt = tokens[i + 1]

                # caso VCV
                if nxt not in VOWELS:

                    # mirar siguiente del siguiente
                    if i + 2 < len(tokens):

                        nxt2 = tokens[i + 2]

                        # VCV → dividir antes de consonante
                        if nxt2 in VOWELS:
                            syllables.append(current)
                            current = []

                        # VCCV
                        else:

                            # coda válida
                            if nxt in CODAS:
                                current.append(nxt)
                                syllables.append(current)
                                current = []
                                i += 1
                            else:
                                syllables.append(current)
                                current = []

                    else:
                        # final
                        if nxt in CODAS:
                            current.append(nxt)
                            i += 1

            syllables.append(current)
            current = []

        i += 1

    if current:
        syllables.append(current)

    #print("SYLLABLES RAW:", syllables)

    return [''.join(s) for s in syllables if len(s) > 0]

In [5]:
#Aplicamos limpieza

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)



In [6]:
from collections import Counter

# =========================
# EXTRAER SUBWORDS SHIPIBO
# =========================

subword_counter = Counter()

for texto in df["source_clean"]:

    palabras = texto.split()

    for palabra in palabras:

        # aplicar silabificador
        silabas = silabificador_shipibo(palabra)

        # contar sílabas/subwords
        subword_counter.update(silabas)

# mostrar más frecuentes
print("\nSUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))


# ==============================================================================
# FILTRAR TOKENS ÚTILES DEL SILABIFICADOR (UMBRAL >= 50)
# ==============================================================================
nuevos_tokens = [token for token, count in subword_counter.items() if count >= 50]
print("Cantidad de nuevos tokens a agregar (freq >= 50):", len(nuevos_tokens))



SUBWORDS MÁS FRECUENTES:
[('i', 49358), ('ja', 35675), ('a', 34753), ('ti', 19882), ('ka', 19741), ('ki', 19239), ('bo', 16370), ('ra', 16368), ('ma', 14632), ('na', 13157), ('bi', 12238), ('yo', 8828), ('ko', 8499), ('to', 8457), ('ke', 7977), ('we', 7876), ('in', 7871), ('o', 7640), ('xon', 7595), ('kin', 7232), ('ba', 7058), ('no', 6979), ('ri', 6827), ('ni', 6104), ('nan', 5949), ('an', 5824), ('jo', 5525), ('jas', 5192), ('ta', 5086), ('ya', 4656), ('tan', 4628), ('e', 4593), ('kon', 4572), ('shi', 4012), ('me', 3892), ('be', 3509), ('pa', 3377), ('non', 3343), ('kan', 3315), ('te', 3266), ('ton', 3131), ('ne', 3088), ('on', 3026), ('di', 2625), ('os', 2533), ('ká', 2506), ('pi', 2332), ('sha', 2286), ('mo', 2129), ('kes', 2069)]
Cantidad de nuevos tokens a agregar (freq >= 50): 261


In [7]:
# =========================
# FILTRAR TOKENS ÚTILES
# =========================

nuevos_tokens = []

for token, freq in subword_counter.items():

    # evitar tokens demasiado pequeños
    if  freq >= 50:

        nuevos_tokens.append(token)

print("Cantidad de nuevos tokens:", len(nuevos_tokens))

print(nuevos_tokens[:50])

Cantidad de nuevos tokens: 261
['ma', 'to', 'ki', 'wi', 'sha', 'bo', 'te', 'xe', 'a', 'ke', 'na', 'we', 'in', 'ba', 'i', 'tan', 'ben', 'chos', 'ko', 'tsin', 'kan', 'ti', 'mo', 'tsa', 'ri', 'an', 'shi', 'nan', 'yo', 'ja', 'kes', 'ka', 'xon', 'me', 'min', 'jas', 'pon', 'pi', 'be', 'ra', 'wen', 'o', 'bi', 'kax', 'nen', 'kon', 'kin', 'nin', 'ta', 'jan']


In [8]:
##Llamar al tokenizador de mBART50

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)


# ==============================================================================
# AGREGAR TOKENS AL TOKENIZER (Aquí se agregan los nuevos tokens al tokenizador)
# ==============================================================================
tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
print("Nuevos tokens reales a agregar:", len(tokens_filtrados))
tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
print("Tokens agregados con éxito:", tokens_agregados)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Nuevos tokens reales a agregar: 65
Tokens agregados con éxito: 65


In [9]:
texto = "matoki wishabo"

print("\nTEXTO DE PRUEBA:")
print(texto)

print("\nTOKENS:")
print(tokenizer.tokenize(texto))

print("\nIDS:")
print(tokenizer(texto)["input_ids"])


TEXTO DE PRUEBA:
matoki wishabo

TOKENS:
['▁mato', 'ki', '▁wish', 'abo']

IDS:
[250004, 72266, 301, 32599, 21552, 2]


In [10]:
texto_prueba = df.iloc[0]["source_clean"]

tokens_silab = tokenizer.tokenize(texto_prueba)

print("\n==============================")
print("TOKENIZACIÓN FINAL")
print("==============================")

print("\nTexto:")
print(texto_prueba)

print("\nTokens:")
print(tokens_silab)

print("\nCantidad:")
print(len(tokens_silab))

print("\nIDs:")
print(tokenizer(texto_prueba)["input_ids"])


TOKENIZACIÓN FINAL

Texto:
matoki wishabo texea

Tokens:
['▁mato', 'ki', '▁wish', 'abo', '▁te', 'xe', 'a']

Cantidad:
7

IDs:
[250004, 72266, 301, 32599, 21552, 120, 4878, 11, 2]


In [11]:
tokenizer.add_tokens(["comeriamos"])
print(tokenizer.tokenize("comeriamos"))

['comeriamos']


***Creación de dataset***

In [12]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 14728
})


In [13]:
print(dataset[0])

{'source': 'matoki wishabo texea', 'target': 'le sobran letras'}


In [14]:
## Función para tokenizar todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [15]:
##Tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)

Map:   0%|          | 0/14728 [00:00<?, ? examples/s]

In [16]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

##Limpiar el dataset de columnas iniciales

tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

***Modelo***

In [17]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Embedding(250120, 1024)

In [18]:
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================

model.resize_token_embeddings(len(tokenizer))

Embedding(250120, 1024)

In [19]:
##Configuración para el entrenamiento

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",              # dónde guardar modelo
    per_device_train_batch_size=1,       # batch pequeño (GPU segura)
    num_train_epochs=3,                  # Modelo ve 6 veces el dataset
    learning_rate=5e-5,                  # estándar para fine-tuning
    save_strategy="epoch",               # guarda cada epoch
    fp16=True                            # usa GPU eficientemente
)

In [20]:
##Creación del trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [21]:
#Entrenar el modelo
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,1.939400
1000,1.272200
1500,1.236700
2000,1.169100
2500,1.136800
3000,1.114600
3500,1.090200
4000,1.076600
4500,1.078800
5000,1.050500


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#sav

TrainOutput(global_step=44184, training_loss=0.6648671971959479, metrics={'train_runtime': 3260.2061, 'train_samples_per_second': 13.553, 'train_steps_per_second': 13.553, 'total_flos': 5984528854155264.0, 'train_loss': 0.6648671971959479, 'epoch': 3.0})

In [22]:
#Guardar el modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4/tokenizer.json')

In [23]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4


In [24]:
def traducir(texto):

    texto = limpiar_texto(texto)

    inputs = tokenizer(
        texto,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=50
    )

    traduccion = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return traduccion

# VALIDACIÓN (val.csv)

In [25]:
#Cargar val

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [26]:
#Preprocesamiento

df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)


In [27]:
#Ejemplos de los resultados

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", df_val.iloc[i]["target_clean"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: jatianra ja dios meniti jawéki min boá jainxon dios yoina menoxontiainkobipari abaini ja joni betan joi benxoax raeananipari mia kati jake jatian moa jaskáa pekáoparikayara min ati jake dios jawéki menikin
REAL: deja allí mismo tu ofrenda ante el altar y vete antes a hacer las paces con tu hermano después vuelve y presenta tu ofrenda
PRED: entonces tú llevas el regalo de dios y vas al al altar de la mesa con el hombre y éste te va a preparar un regalo de dios después de esto serás premiado
-----
INPUT: jain yoyo ati kirika benxoati
REAL: biblioteca del aula
PRED: biblioteca de aula
-----
INPUT: teayamawe yoyo ikashamaitian
REAL: no lo obligues
PRED: si respetan el turno
-----
INPUT: moara jaton aká jawékibo jan jato yoixonti nete senenke
REAL: adoren al que hizo el cielo la tierra el mar y los manantiales de agua
PRED: ha llegado la hora de juzgar
-----
INPUT: rotacion itan traslacion
REAL: rotación y traslación
PRED: rotación y traslación
-----


# Métricas para val.csv

In [28]:
#Preparar listas

preds = []
refs = []

for i in range(len(df_val)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = df_val.iloc[i]["target_clean"]

    preds.append(pred)
    refs.append([ref])

In [29]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")
print("BLEU:", bleu.compute(predictions=preds, references=refs))

BLEU: {'bleu': 0.12386775389904446, 'precisions': [0.3868831446272164, 0.16620293081061027, 0.09166962429650662, 0.056107083100948134], 'brevity_penalty': 0.9185289633702736, 'length_ratio': 0.9216744112782547, 'translation_length': 23405, 'reference_length': 25394}


In [30]:
#CHRF

chrf = evaluate.load("chrf")
print("ChrF:", chrf.compute(predictions=preds, references=refs))

ChrF: {'score': 34.23664656061005, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [31]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(df_val)):
    data.append({
        "src": df_val.iloc[i]["source"],  # ORIGINAL
        "mt": preds[i],
        "ref": df_val.iloc[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8, gpus=1)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/437 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') th

COMET: -0.5712760258890943


In [32]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: {'meteor': 0.32252908422147375}


In [33]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Promedio): 0.7871698825855349
BERTScore Recall (Promedio): 0.7779654002526349
BERTScore F1 (Promedio): 0.7821926403991041


In [34]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL:
# 1. learning_rate=3e-5 (Se reduce a 3e-5 en comparación con el valor inicial de 5e-5)
#    para realizar un ajuste fino más conservador y evitar el olvido catastrófico de los pesos
#    ya optimizados en la fase 1.
# 2. num_train_epochs = 3 (Se mantiene en 3 épocas para el refinamiento adicional).
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Lote efectivo de 4 (mayor estabilidad)
    num_train_epochs=3,
    learning_rate=3e-5,
    weight_decay=0.01,               # Evita overfitting
    warmup_ratio=0.1,                # Calentamiento del 10% de pasos
    label_smoothing_factor=0.1,      # Regularización para NMT
    save_strategy="epoch",
    fp16=True
)



In [35]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [36]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [37]:
from transformers import AutoModelForSeq2SeqLM
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4


Embedding(250120, 1024, padding_idx=1)

In [38]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [39]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
500,4.001100
1000,3.105600
1500,3.067400
2000,3.010600
2500,2.998700
3000,2.959100
3500,2.949600
4000,2.847200
4500,2.807800
5000,2.830400


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#sav

TrainOutput(global_step=11046, training_loss=2.8649161313077838, metrics={'train_runtime': 2104.356, 'train_samples_per_second': 20.996, 'train_steps_per_second': 5.249, 'total_flos': 5984528854155264.0, 'train_loss': 2.8649161313077838, 'epoch': 3.0})

In [40]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mbart_ShipiboKonibo_v4_retrained/tokenizer.json')

In [42]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos


val_dataset = Dataset.from_pandas(df_val)

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: jatianra ja dios meniti jawéki min boá jainxon dios yoina menoxontiainkobipari abaini ja joni betan joi benxoax raeananipari mia kati jake jatian moa jaskáa pekáoparikayara min ati jake dios jawéki menikin
REAL: deja allí mismo tu ofrenda ante el altar y vete antes a hacer las paces con tu hermano después vuelve y presenta tu ofrenda
PRED: y si el el eldodododo al al al al al al al al al y al al al al al al y después
-----
INPUT: jain yoyo ati kirika benxoati
REAL: biblioteca del aula
PRED: biblioteca biblioteca biblioteca
-----
INPUT: teayamawe yoyo ikashamaitian
REAL: no lo obligues
PRED: no te preocupes cuando nos
-----
INPUT: moara jaton aká jawékibo jan jato yoixonti nete senenke
REAL: adoren al que hizo el cielo la tierra el mar y los manantiales de agua
PRED: ha llegado el día del juicio
-----
INPUT: rotacion itan traslacion
REAL: rotación y traslación
PRED: rotación y traslación
-----


In [43]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [44]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.041006758115764284,
 'precisions': [0.23917354609601993,
  0.06914625541780063,
  0.026923503941378928,
  0.012915129151291513],
 'brevity_penalty': 0.8373892877318512,
 'length_ratio': 0.8492812659120285,
 'translation_length': 21683,
 'reference_length': 25531}

In [45]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 22.429071433212243, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [46]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.44001773810788125


In [47]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR (Validación - Reentrenado): {'meteor': 0.18211577112801364}


In [48]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Validación - Reentrenado - Promedio): 0.6890512737063594
BERTScore Recall (Validación - Reentrenado - Promedio): 0.6929821566289563
BERTScore F1 (Validación - Reentrenado - Promedio): 0.69039259510219


In [49]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [50]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [51]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 1841
})


In [52]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [53]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.044847892072161846,
 'precisions': [0.24732953743353253,
  0.07635239567233384,
  0.030203060121722313,
  0.013335018643746446],
 'brevity_penalty': 0.8539967704083944,
 'length_ratio': 0.863686242633611,
 'translation_length': 21251,
 'reference_length': 24605}

In [54]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 22.512933314146476, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [55]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.4444344032400008


In [56]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR: {'meteor': 0.19852668488416253}


In [57]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Promedio): 0.6920520994308395
BERTScore Recall (Promedio): 0.6978308213333407
BERTScore F1 (Promedio): 0.6943452360674067
